In [2]:
import sys
sys.path.append("scripts/") 

In [3]:
import os
import sys
import numpy as np
from time import time
import xgboost as xgb
xgb.set_config(verbosity=0)
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt

In [4]:
def cov_matrix_vectorized(x):
    # x should be of shape (timesteps, features) or (features, timesteps)
    if x.shape[0] < x.shape[1]:
        x = x.T
    cov = np.cov(x, rowvar=False)
    return cov.flatten()

In [5]:
ML_DATA_PATH = "/project/scratch/p200631/Silvana/gpu_utilization/results/60-random/ml.npz"

In [6]:
ml_data = np.load(ML_DATA_PATH)
X_train, y_train, X_test, y_test = ml_data['X_train'], ml_data['y_train'],ml_data['X_test'],ml_data['y_test']

In [7]:
from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, 
                             test_size=0.2,
                             random_state=37)

for train_index, valid_index in sss.split(X_train, y_train):
    X_train1, X_valid = X_train[train_index], X_train[valid_index]
    y_train1, y_valid = y_train[train_index], y_train[valid_index]
    model_train, model_valid = model[train_index], model[valid_index]
print(X_train.shape,y_train.shape,model_train.shape)
print(X_valid.shape,y_valid.shape,model_valid.shape)

NameError: name 'model' is not defined

In [ ]:
# standardize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
X_test = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

In [ ]:
# covariance
X_train = np.array(list(map(cov_matrix_vectorized,X_train)))
X_test = np.array(list(map(cov_matrix_vectorized,X_test)))

In [ ]:
xgb_clf = xgb.XGBClassifier(learning_rate=0.1, 
                            objective='multi:softmax',
                            silent=True, 
                            num_boost_round=3, 
                            early_stopping_rounds=10,
                            max_depth=3,
                            nthread=1,
                            num_class=np.unique(y_train).shape[0],
                            label_encoder=False)


In [ ]:
xgb_params = {
    'gamma':[0.05,0.5,1],
    'alpha':[0.05,0.5,1],
    'lambda':[0.05,0.5,1]    
}


In [ ]:
print('*******************')
print('Running Grid Search')
print('*******************')

grid_search = GridSearchCV(estimator=xgb_clf, 
                           param_grid=xgb_params,
                           n_jobs=-1,
                           verbose=1,
                           scoring='accuracy',
                           cv=10)
      
# Fit the GridSearch on the training data
t0 = time()
grid_search.fit(X_train,y_train)
print('Done in {:0.3f}s\n'.format(time()-t0))

print('Best training score: {:0.4f}\n'.format(grid_search.best_score_))
best_params = grid_search.best_estimator_.get_params()
print('Best parameters:\n')
for param_name in sorted(xgb_params.keys()):
    print('  {}: {}'.format(param_name, best_params[param_name]))
print('\nTest set accuracy using best hyperparameters {:0.4f}\n'.format(grid_search.score(X_test,y_test)))


In [ ]:
####################
# Feature Importance
####################

best_estimator = grid_search.best_estimator_
feature_importance = best_estimator.feature_importances_

feature_importance_matrix = np.zeros((7,7))
feature_importance_matrix[np.triu_indices(7)] = feature_importance

features = [
    '',
    'utilization gpu pct',
    'utilization memory pct',
    'memory free MiB',
    'memory used MiB',
    'temperature gpu',
    'temperature memory',
    'power draw W'
]
plt.set_cmap(cmap='inferno')
fig = plt.figure()
ax = fig.add_subplot(111)
ax.set_xticklabels(features,rotation=90)
ax.set_yticklabels(features,rotation=None)
cax = ax.matshow(feature_importance_matrix, 
                 interpolation='nearest')
fig.colorbar(cax)
plt.savefig(os.path.join(RESULTS_PATH,'xgb-feature-importance-{}.jpg'.format(DATA_FILE)),
            bbox_inches='tight',
            dpi=150)
plt.close()
